In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

# QUICK=True re-runs a fast, few-seed version of an experiment in-kernel (serial, no Pool);
# the default (QUICK=False) REPLAYS the committed 20-seed grid so every figure renders
# instantly. Heavy studies (exp14 at H=512; the torch-dependent hybrid front-end) always
# default to replay and document a `--full` shell command instead.
QUICK = False

GREEN, INDIGO, RED, GOLD, GREY = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2"
PURPLE, ORANGE, INK = "#b07cc6", "#e0a93b", "#2b2b2b"  # DFA feedback / homeostasis / ink

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

print("data/results:", paths.results_dir())

In [ ]:
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle

def _compound_node(ax, cx, cy, s=0.085):
    # compound crosspoint: left half = non-volatile weight device, right half = trace device
    ax.add_patch(Rectangle((cx - s, cy - s), s, 2 * s, fc="#cbd6e0", ec=INK, lw=0.9, zorder=4))
    ax.add_patch(Rectangle((cx, cy - s), s, 2 * s, fc="white", ec=INDIGO, lw=1.0, zorder=4))

fig, ax = plt.subplots(figsize=(7.6, 5.0))
ax.set_xlim(0, 10); ax.set_ylim(-0.3, 7.6); ax.axis("off")

# global reward broadcast (top, red dashed)
ax.plot([0.7, 9.4], [7.05, 7.05], color=RED, ls="--", lw=1.6, zorder=2)
ax.text(5.05, 7.32, r"global third factor $(R-b)$ broadcast to every crosspoint",
        color=RED, fontsize=9.5, ha="center")

# layer 1: inputs (rows) x hidden (cols)  ->  W1
in_y = [6.15, 5.5]; h_x = [2.3, 3.1, 3.9]; h_lif_y = 4.05
ax.text(0.15, 5.82, "state\ninputs", color=GREEN, fontsize=9, va="center")
for x in h_x:
    ax.plot([x, x], [6.45, h_lif_y + 0.2], color=INDIGO, lw=1.4, zorder=1)
for y in in_y:
    ax.add_patch(Circle((1.25, y), 0.11, fc="white", ec=GREEN, lw=1.6, zorder=4))
    ax.plot([1.36, 4.25], [y, y], color=GREEN, lw=2.0, zorder=2)
for y in in_y:
    for x in h_x:
        _compound_node(ax, x, y, s=0.085)
ax.text(3.1, 6.62, r"$W_1$ (input$\rightarrow$hidden array)", color=INDIGO, fontsize=8.5, ha="center")

# hidden LIF layer (with homeostasis rings)
for x in h_x:
    ax.add_patch(Circle((x, h_lif_y), 0.32, fc="none", ec=ORANGE, lw=1.3, ls=(0, (2, 1.5)), zorder=4))
    ax.add_patch(Circle((x, h_lif_y), 0.20, fc="#eaf0fb", ec=INDIGO, lw=1.5, zorder=5))
    ax.text(x, h_lif_y, "LIF", color=INDIGO, fontsize=6.5, ha="center", va="center")
ax.text(4.35, h_lif_y, "hidden\nneurons", color=INDIGO, fontsize=8.5, va="center")
ax.annotate("", xy=(1.85, h_lif_y + 0.05), xytext=(1.1, h_lif_y + 0.05),
            arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1.5))
ax.text(0.95, h_lif_y + 0.05, "local\nhomeostasis", color=ORANGE, fontsize=8.0, ha="center", va="center")

# layer 2: hidden (rows) x action (cols)  ->  W2
a_x = [6.4, 7.2]; row2_y = [2.45, 2.05, 1.65]; bus_x = 5.55
for x in h_x:
    ax.plot([x, bus_x], [h_lif_y - 0.32, h_lif_y - 0.32], color=INDIGO, lw=1.0, zorder=1)
ax.plot([bus_x, bus_x], [h_lif_y - 0.32, row2_y[-1]], color=INDIGO, lw=1.0, zorder=1)
for y in row2_y:
    ax.plot([bus_x, 7.6], [y, y], color=INDIGO, lw=1.4, zorder=1)
for x in a_x:
    ax.plot([x, x], [2.7, 0.55], color=INDIGO, lw=1.4, zorder=1)
    for y in row2_y:
        _compound_node(ax, x, y, s=0.080)
ax.text(6.8, 2.78, r"$W_2$ (hidden$\rightarrow$action array)", color=INDIGO, fontsize=8.5, ha="center")
ax.add_patch(Rectangle((8.15, 6.05), 0.14, 0.28, fc="#cbd6e0", ec=INK, lw=0.8, zorder=5))
ax.add_patch(Rectangle((8.29, 6.05), 0.14, 0.28, fc="white", ec=INDIGO, lw=0.9, zorder=5))
ax.text(8.55, 6.19, "compound cell:\nweight | trace device", color=INK, fontsize=6.6, va="center")

# action LIF + WTA
for x in a_x:
    ax.add_patch(Circle((x, 0.35), 0.20, fc="#eaf0fb", ec=INDIGO, lw=1.5, zorder=5))
    ax.text(x, 0.35, "LIF", color=INDIGO, fontsize=6.5, ha="center", va="center")
ax.text(7.65, 0.35, "action\nneurons", color=INDIGO, fontsize=8.5, va="center")

# DFA fixed-random feedback (purple) -- no W2^T transport
ax.add_patch(FancyArrowPatch((6.55, 0.15), (3.1, 3.55), connectionstyle="arc3,rad=0.30",
             arrowstyle="-|>", mutation_scale=13, color=PURPLE, lw=1.9, ls=(0, (5, 2)), zorder=2))
ax.text(3.05, 1.15, r"fixed random feedback $B$:  $L_h = B\,L_a$" "\n"
        r"per-neuron credit, no $W_2^{\top}$ transport",
        color=PURPLE, fontsize=8.0, ha="center", va="center")
plt.show()

In [ ]:
if QUICK:
    from mrl_trace.deep import run_deep_local
    r = run_deep_local(seeds=4, trials=1200)      # fast few-seed recompute; same grid schema
else:
    r = paths.load_result("exp7_deep_local.npy")

ORDER = ["shallow", "elm", "global", "dfa", "no_trace", "dfa_homeo"]
LAB = {"shallow": "shallow\n(1 layer)", "elm": "ELM\n(fixed hidden)",
       "global": "global\nscalar", "dfa": "DFA", "no_trace": "no-trace",
       "dfa_homeo": "DFA +\nhomeostasis"}
COL = {"shallow": GREY, "elm": GOLD, "global": RED, "dfa": INDIGO,
       "no_trace": "#c9ced6", "dfa_homeo": GREEN}
finals, ci, curves = r["finals"], r["ci"], r["curves"]
chance, crit = r["chance"], r["crit"]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.2, 3.9),
                               gridspec_kw={"width_ratios": [1.4, 1.0]})
# (a) learning curves (already stored as seed-mean running rate per trial)
for k in ("dfa_homeo", "dfa", "global", "no_trace", "elm", "shallow"):
    c = np.asarray(curves[k], float)
    w = max(1, len(c) // 60)
    rc = np.convolve(c, np.ones(w) / w, mode="valid")
    axA.plot(np.arange(len(rc)), rc, color=COL[k], lw=1.7, label=LAB[k].replace("\n", " "))
axA.axhline(crit, ls=":", color=GREY, lw=1.0); axA.axhline(chance, ls="--", color=RED, lw=1.0)
axA.set_xlabel("trial"); axA.set_ylabel("reward rate (running)"); axA.set_ylim(0.28, 1.04)
axA.set_title("(a) deep-XOR learning curves", fontsize=10, loc="left")
axA.legend(fontsize=6.6, ncol=2, frameon=False, loc="lower right"); _clean(axA)
# (b) final reward rate by condition with bootstrap CI
x = np.arange(len(ORDER))
means = [np.asarray(finals[k]).mean() for k in ORDER]
err = [[m - ci[k][0] for m, k in zip(means, ORDER)], [ci[k][1] - m for m, k in zip(means, ORDER)]]
axB.bar(x, means, 0.66, color=[COL[k] for k in ORDER], yerr=err, capsize=3, error_kw=dict(lw=0.9))
axB.axhline(crit, ls=":", color=GREY, lw=1.0); axB.axhline(chance, ls="--", color=RED, lw=1.0)
axB.set_xticks(x); axB.set_xticklabels([LAB[k] for k in ORDER], fontsize=6.8)
axB.set_ylabel("final reward rate"); axB.set_ylim(0, 1.1)
axB.set_title("(b) final rate", fontsize=10, loc="left"); _clean(axB)
plt.show()

print("finals:", {k: round(float(np.asarray(finals[k]).mean()), 3) for k in ORDER})
print("pre-registered criteria:", {k: ("PASS" if v else "fail") for k, v in r["criteria"].items()})
print("  (C3 fail = plain DFA 0.73 < 0.75, bimodal; C6 PASS = homeostasis fix reaches ceiling)")

In [ ]:
if QUICK:
    from mrl_trace.deep import run_deep_dms
    r = run_deep_dms(seeds=4, trials=1200)        # fast few-seed recompute; same grid schema
else:
    r = paths.load_result("exp13_deep_dms.npy")

LABELS = {"dfa_homeo_dist": "DFA + homeostasis (distractor)",
          "dfa_homeo_nodist": "DFA + homeostasis (no distractor)",
          "dfa_dist": "DFA, no homeostasis (distractor)",
          "no_trace_dist": "no-trace control (distractor)"}
COL = {"dfa_homeo_dist": GREEN, "dfa_homeo_nodist": INDIGO, "dfa_dist": GOLD, "no_trace_dist": GREY}
ORDER = ["dfa_homeo_dist", "dfa_homeo_nodist", "dfa_dist", "no_trace_dist"]
curves, finals, ci = r["curves"], r["finals"], r["ci"]
solved, seeds = r["seeds_solved"], r.get("seeds", 20)
chance, crit = r.get("chance", 0.5), r.get("crit", 0.75)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 3.9),
                               gridspec_kw={"width_ratios": [1.5, 1.0]})
# (a) learning curves (seed-mean per trial -> running mean, window 50)
for k in ORDER:
    c = np.asarray(curves[k], float)
    w = max(1, min(50, len(c) // 20))
    rc = np.convolve(c, np.ones(w) / w, mode="valid")
    axA.plot(np.arange(len(rc)), rc, color=COL[k], lw=1.8, label=LABELS[k])
axA.axhline(chance, ls="--", color=RED, lw=1.1); axA.axhline(crit, ls=":", color=GREY, lw=1.0)
axA.set_xlabel("trial"); axA.set_ylabel("reward rate"); axA.set_ylim(0.3, 1.04)
axA.set_title("(a) deep DMS + distractor", fontsize=10, loc="left")
axA.legend(fontsize=7.0, ncol=1, frameon=False, loc="lower right"); _clean(axA)
# (b) final reward rate bars with CI + seeds-solved
x = np.arange(len(ORDER))
means = [np.asarray(finals[k]).mean() for k in ORDER]
los = [means[i] - ci[k][0] for i, k in enumerate(ORDER)]
his = [ci[k][1] - means[i] for i, k in enumerate(ORDER)]
axB.bar(x, means, 0.62, color=[COL[k] for k in ORDER], yerr=[los, his], capsize=3, error_kw=dict(lw=0.9))
for i, k in enumerate(ORDER):
    axB.text(i, min(means[i] + his[i] + 0.04, 1.05), f"{solved[k]}/{seeds}",
             ha="center", va="bottom", fontsize=8)
axB.axhline(chance, ls="--", color=RED, lw=1.1); axB.axhline(crit, ls=":", color=GREY, lw=1.0)
axB.set_xticks(x)
axB.set_xticklabels(["homeo\n+dist", "homeo\nno-dist", "no-homeo\n+dist", "no-trace"], fontsize=7.5)
axB.set_ylabel("final reward rate"); axB.set_ylim(0, 1.12)
axB.set_title("(b) final rate", fontsize=10, loc="left"); _clean(axB)
plt.show()

print("finals:", {k: round(float(np.asarray(finals[k]).mean()), 3) for k in ORDER})
print("pre-registered criteria:", {k: ("PASS" if v else "fail") for k, v in r["criteria"].items()})

In [ ]:
r = paths.load_result("exp14_array_scale.npy")            # full PF grid: H={32,512}, p={0,0.5}
rs = paths.load_result("exp14_array_scale_sweep.npy")     # finer sweep: H={8,32,128,512}, p={0,..,0.5}
H, P, grid, ctrl = np.asarray(r["H"]), np.asarray(r["p"]), np.asarray(r["grid"]), np.asarray(r["ctrl"])
Hs, Ps, grids, ctrls = (np.asarray(rs["H"]), np.asarray(rs["p"]),
                        np.asarray(rs["grid"]), np.asarray(rs["ctrl"]))

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.4, 3.9))
# (a) the coarse full-PF grid: full stack (teal shades by fault level) vs no-trace, per H
pcol = {0.0: GREEN, 0.5: "#7fc4a8"}
xw = np.arange(len(H)); w = 0.8 / (len(P) + 1)
for j, p in enumerate(P):
    m = grid[:, j, 0]; lo = m - grid[:, j, 1]; hi = grid[:, j, 2] - m
    axA.bar(xw + (j - len(P) / 2) * w, m, w, color=pcol.get(float(p), GREEN),
            yerr=[lo, hi], capsize=2.5, error_kw=dict(lw=0.9), label=f"full stack, {p:.0%} stuck")
ntc = ctrl.mean(1)   # no-trace collapses over p (sits at chance regardless)
axA.bar(xw + (len(P) - len(P) / 2) * w, ntc, w, color=GREY, label="no-trace control")
axA.axhline(0.5, ls="--", color=RED, lw=1.1, label="chance $1/A$")
axA.axhline(0.75, ls=":", color=GREY, lw=1.0)
axA.set_xticks(xw); axA.set_xticklabels([f"$H={h}$" for h in H])
axA.set_xlabel("hidden width $H$"); axA.set_ylabel("final reward rate"); axA.set_ylim(0, 1.05)
axA.set_title("(a) full PF fault stack", fontsize=10, loc="left")
axA.legend(fontsize=6.6, ncol=2, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.06)); _clean(axA)
# (b) the finer sweep as a heat-map of the full-stack mean over (H, p)
im = axB.imshow(grids[:, :, 0], cmap="viridis", vmin=0.5, vmax=1.0, aspect="auto", origin="lower")
axB.set_xticks(range(len(Ps))); axB.set_xticklabels([f"{p:.0%}" for p in Ps])
axB.set_yticks(range(len(Hs))); axB.set_yticklabels([f"{h}" for h in Hs])
axB.set_xlabel("stuck-off fraction $p$"); axB.set_ylabel("hidden width $H$")
axB.set_title("(b) full-stack mean rate sweep", fontsize=10, loc="left")
for i in range(len(Hs)):
    for j in range(len(Ps)):
        axB.text(j, i, f"{grids[i, j, 0]:.2f}", ha="center", va="center",
                 fontsize=7.5, color="white" if grids[i, j, 0] < 0.8 else "0.1")
cb = fig.colorbar(im, ax=axB, fraction=0.046, pad=0.04); cb.set_label("final reward rate", fontsize=8)
plt.show()

print("full PF grid faults:", r["faults"], f"(seeds={r['seeds']})")
print("sweep faults:", rs["faults"], f"(seeds={rs['seeds']})")
print("no-trace control range (sweep):", round(float(ctrls.min()), 3), "-", round(float(ctrls.max()), 3),
      "-> stays at chance; full stack learns at every H, p with disjoint CIs")

In [ ]:
r = paths.load_result("tier6_results.npy")
res = r["results"]; chance, crit = r["chance"], r["crit"]
order = ["hybrid", "raw", "no_trace"]
labels = ["hybrid\n(front-end)", "raw\npixels", "no-\ntrace"]
COL = {"hybrid": GREEN, "raw": GREY, "no_trace": "#c9ced6"}
# result tuples are (mean, sd, lo, hi); error bars are the bootstrap 95% CI
vals = np.array([res[k][0] for k in order])
lo = np.array([res[k][2] for k in order]); hi = np.array([res[k][3] for k in order])
err = np.vstack([vals - lo, hi - vals])

fig, ax = plt.subplots(figsize=(5.6, 3.9))
x = np.arange(len(order))
ax.bar(x, vals, 0.6, color=[COL[k] for k in order], yerr=err, capsize=3,
       edgecolor="white", error_kw=dict(lw=0.9), zorder=3)
ax.axhline(crit, ls=":", color=GREY, lw=1.2, label="criterion")
ax.axhline(chance, ls="--", color=RED, lw=1.1, label="chance $1/A$")
ax.text(0, hi[0] + 0.03, "PASS", fontsize=9, ha="center", color=GREEN, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("final reward rate"); ax.set_ylim(0, 0.82)
ax.set_title(f"Hybrid stack: front-end acc {r['front_acc']:.2f}  (C={r['C']}, A={r['A']}, P={r['P']})",
             fontsize=9.5)
ax.legend(fontsize=8, frameon=False, loc="upper right"); _clean(ax)
plt.show()

print("finals (mean):", {k: round(float(res[k][0]), 3) for k in order})
print("front-end accuracy:", r["front_acc"], "| seeds:", r["n_seeds"])
print("criteria:", {"C1": r["C1"], "C2": r["C2"], "C3": r["C3"]},
      "-> hybrid reaches criterion; raw-pixel + no-trace stay at chance")

In [ ]:
# Uncomment to launch a full sweep as a subprocess (streamed; heavy -- minutes each):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "mrl_trace.deep", "--exp7", "--full"])
print("see the markdown above for the full-scale commands")